# ⏱️ Hourly Exploratory Data Analysis & Bottleneck Profiling
This notebook analyzes:
1. **Hourly Scheduled Capacity:** GTFS `stop_times.txt` aggregated across hours (0–23)
2. **Arterial Speed Degradation:** Chicago Traffic Tracker segment speeds by hour
3. **Headway Regularity & Bus Bunching:** Comparing peak hours (7–9 AM, 4–6 PM) against off-peak periods
4. **Spatial Bottleneck Mapping:** High-delay corridor detection

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath('..'))
from src.features import generate_synthetic_hourly_history
from src.hourly_aggregator import build_hourly_traffic_profiles

## 1. Load Hourly Transit & Traffic History

In [ ]:
df_hourly = generate_synthetic_hourly_history(num_days=30)
print(f"Loaded {len(df_hourly)} hourly observations across routes: {df_hourly['route'].unique()}")
display(df_hourly.head())

## 2. Diurnal Arterial Speed Profile (Chicago Traffic Tracker)

In [ ]:
hourly_summary = df_hourly.groupby('hour').agg({
    'avg_speed_mph': ['mean', 'min', 'max'],
    'delay_min': 'mean',
    'scheduled_headway_min': 'mean'
}).reset_index()

fig, ax1 = plt.subplots(figsize=(10, 5))
color = 'tab:blue'
ax1.set_xlabel('Hour of Day (0–23)', fontsize=12)
ax1.set_ylabel('Traffic Speed (mph)', color=color, fontsize=12)
ax1.plot(hourly_summary['hour'], hourly_summary['avg_speed_mph']['mean'], color=color, marker='o', linewidth=2.5, label='Avg Speed')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, linestyle='--', alpha=0.4)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Transit Delay (minutes)', color=color, fontsize=12)
ax2.bar(hourly_summary['hour'], hourly_summary['delay_min']['mean'], color=color, alpha=0.35, width=0.6, label='Avg Delay')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Chicago Arterial Traffic Speed vs Transit Delay by Hour of Day', fontsize=14)
plt.show()

## 3. Delay Distribution Across Peak and Off-Peak Hours

In [ ]:
def classify_period(h):
    if 7 <= h <= 9:
        return 'Morning Peak (7-9 AM)'
    elif 16 <= h <= 18:
        return 'Evening Peak (4-6 PM)'
    elif 10 <= h <= 15:
        return 'Midday (10 AM-3 PM)'
    else:
        return 'Night / Early Morning'

df_hourly['period'] = df_hourly['hour'].apply(classify_period)

plt.figure(figsize=(10, 5))
sns.boxplot(data=df_hourly, x='period', y='delay_min', palette='Set2')
plt.title('Bus Delay Distribution by Time-of-Day Period')
plt.ylabel('Delay (minutes)')
plt.xlabel('Period')
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()